### Setup

In [0]:
print("gold_layer_started")

from pyspark.sql import functions as f
from delta.tables import DeltaTable

### Use Catalog

In [0]:
%sql
USE CATALOG marketing_analytics_capstone;
CREATE SCHEMA IF NOT EXISTS gold;

### Read Silver

In [0]:
df = spark.read.table("marketing_analytics_capstone.silver.silver_campaigns") \
    .filter("dq_pass = true")

### DIMENSION TABLES

#### dim_campaign

In [0]:
dim_campaign = (
    df.select(
        "campaign_id",
        "campaign_type",
        "target_audience",
        "customer_segment"
    )
    .dropDuplicates(["campaign_id"])
)

#### Incremental MERGE

In [0]:
target_table = "marketing_analytics_capstone.gold.dim_campaign"

if spark.catalog.tableExists(target_table):
    target = DeltaTable.forName(spark, target_table)
    target.alias("t").merge(
        dim_campaign.alias("s"),
        "t.campaign_id = s.campaign_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    dim_campaign.write.saveAsTable(target_table)

#### dim_channel

In [0]:
dim_channel = df.select("channel_used").dropDuplicates()

dim_channel.write.mode("overwrite").saveAsTable(
    "marketing_analytics_capstone.gold.dim_channel"
)

#### dim_company

In [0]:
dim_company = df.select("company").dropDuplicates()

dim_company.write.mode("overwrite").saveAsTable(
    "marketing_analytics_capstone.gold.dim_company"
)

#### dim_location

In [0]:
dim_location = df.select("location").dropDuplicates()

dim_location.write.mode("overwrite").saveAsTable(
    "marketing_analytics_capstone.gold.dim_location"
)

#### dim_date

In [0]:
dim_date = (
    df.select("date")
    .dropDuplicates()
    .withColumn("year", f.year("date"))
    .withColumn("month", f.month("date"))
    .withColumn("month_name", f.date_format("date", "MMMM"))
    .withColumn("quarter", f.quarter("date"))
)

dim_date.write.mode("overwrite").saveAsTable(
    "marketing_analytics_capstone.gold.dim_date"
)

### FACT TABLE

#### Build Fact Data

In [0]:
fact_df = df.select(
    "campaign_id",
    "date",
    "channel_used",
    "company",
    "location",
    "clicks",
    "impressions",
    "conversion_rate",
    "acquisition_cost",
    "roi",
    "engagement_score",
    "ctr"
)

#### Aggregate Fact

In [0]:
fact_agg = (
    fact_df.groupBy(
        "campaign_id",
        "date",
        "channel_used",
        "company",
        "location"
    )
    .agg(
        f.sum("clicks").alias("total_clicks"),
        f.sum("impressions").alias("total_impressions"),
        f.sum("acquisition_cost").alias("total_cost"),
        f.avg("conversion_rate").alias("avg_conversion_rate"),
        f.avg("roi").alias("avg_roi"),
        f.avg("ctr").alias("avg_ctr")
    )
)

### MERGE into Fact Table (Incremental)

In [0]:
target_table = "marketing_analytics_capstone.gold.fact_campaign_performance"

if spark.catalog.tableExists(target_table):
    target = DeltaTable.forName(spark, target_table)
    target.alias("t").merge(
        fact_agg.alias("s"),
        """
        t.campaign_id = s.campaign_id AND
        t.date = s.date AND
        t.channel_used = s.channel_used AND
        t.company = s.company AND
        t.location = s.location
        """
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    fact_agg.write.partitionBy("date").saveAsTable(target_table)

### PRE-AGGREGATED KPI TABLES

#### KPI – Daily Performance

In [0]:
kpi_daily = (
    df.groupBy("date")
    .agg(
        f.sum("clicks").alias("total_clicks"),
        f.sum("impressions").alias("total_impressions"),
        f.sum("acquisition_cost").alias("total_cost"),
        f.avg("ctr").alias("avg_ctr"),
        f.avg("roi").alias("avg_roi")
    )
)

kpi_daily.write.mode("overwrite").saveAsTable(
    "marketing_analytics_capstone.gold.kpi_daily_campaign_performance"
)

#### KPI – Channel Performance

In [0]:
kpi_channel = (
    df.groupBy("date", "channel_used")
    .agg(
        f.sum("clicks").alias("total_clicks"),
        f.sum("impressions").alias("total_impressions"),
        f.avg("ctr").alias("avg_ctr"),
        f.avg("roi").alias("avg_roi")
    )
)

kpi_channel.write.mode("overwrite").saveAsTable(
    "marketing_analytics_capstone.gold.kpi_channel_performance"
)

#### KPI – Campaign Summary

In [0]:
kpi_campaign = (
    df.groupBy("campaign_id")
    .agg(
        f.sum("clicks").alias("total_clicks"),
        f.sum("impressions").alias("total_impressions"),
        f.sum("acquisition_cost").alias("total_cost"),
        f.avg("roi").alias("avg_roi")
    )
)

kpi_campaign.write.mode("overwrite").saveAsTable(
    "marketing_analytics_capstone.gold.kpi_campaign_summary"
)

#### KPI – Monthly Trends

In [0]:
kpi_monthly = (
    df.withColumn("year", f.year("date"))
      .withColumn("month", f.month("date"))
      .groupBy("year", "month")
      .agg(
          f.sum("clicks").alias("total_clicks"),
          f.sum("impressions").alias("total_impressions"),
          f.avg("ctr").alias("avg_ctr"),
          f.avg("roi").alias("avg_roi")
      )
)

kpi_monthly.write.mode("overwrite").saveAsTable(
    "marketing_analytics_capstone.gold.kpi_monthly_trends"
)

### Validation

In [0]:
%sql
SELECT COUNT(*) FROM marketing_analytics_capstone.gold.fact_campaign_performance;

In [0]:
%sql
SELECT * FROM marketing_analytics_capstone.gold.kpi_daily_campaign_performance LIMIT 10;